# Notebook 01 – Exploratory Data Analysis
**DSA 210 – Social Media Use & Mental Health**

**Dataset:** Social Media & Mental Health Survey (`smmh.csv`) — 481 real survey responses collected April–May 2022.

This notebook explores the dataset to understand distributions, usage patterns, demographic breakdowns, and correlations between social media behavior and mental health indicators before any formal testing.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
os.makedirs("../figures", exist_ok=True)
pd.set_option("display.max_columns", None)

# ── Column shortnames (mirrors data_cleaner.py) ──────────────────────────────
MENTAL_HEALTH_COLS = [
    "purposeless_use", "distraction_busy", "restlessness",
    "easily_distracted", "bothered_by_worries", "difficulty_concentrating",
    "compare_to_others", "feeling_about_comparisons", "seek_validation",
    "feel_depressed", "interest_fluctuation", "sleep_issues",
]

USAGE_MAP = {
    "Less than an Hour":      0.5,
    "Between 1 and 2 hours":  1.5,
    "Between 2 and 3 hours":  2.5,
    "Between 3 and 4 hours":  3.5,
    "Between 4 and 5 hours":  4.5,
    "More than 5 hours":      6.0,
}

COL_RENAME = {
    "1. What is your age?": "age",
    "2. Gender": "gender",
    "3. Relationship Status": "relationship_status",
    "4. Occupation Status": "occupation",
    "5. What type of organizations are you affiliated with?": "org_type",
    "6. Do you use social media?": "uses_social_media",
    "7. What social media platforms do you commonly use?": "platforms",
    "8. What is the average time you spend on social media every day?": "daily_usage",
    "9. How often do you find yourself using Social media without a specific purpose?": "purposeless_use",
    "10. How often do you get distracted by Social media when you are busy doing something?": "distraction_busy",
    "11. Do you feel restless if you haven't used Social media in a while?": "restlessness",
    "12. On a scale of 1 to 5, how easily distracted are you?": "easily_distracted",
    "13. On a scale of 1 to 5, how much are you bothered by worries?": "bothered_by_worries",
    "14. Do you find it difficult to concentrate on things?": "difficulty_concentrating",
    "15. On a scale of 1-5, how often do you compare yourself to other successful people through the use of social media?": "compare_to_others",
    "16. Following the previous question, how do you feel about these comparisons, generally speaking?": "feeling_about_comparisons",
    "17. How often do you look to seek validation (Likes, Comments etc) from social media?": "seek_validation",
    "18. How often do you feel depressed or low?": "feel_depressed",
    "19. On a scale of 1 to 5, how frequently does your interest in daily activities fluctuate?": "interest_fluctuation",
    "20. On a scale of 1 to 5, how often do you face issues regarding sleep?": "sleep_issues",
}

print("Libraries loaded ✓")

## 1. Load & Clean Data
Load `smmh.csv`, rename columns, filter social media users, and engineer features.

In [ ]:
raw = pd.read_csv("../data/raw/smmh.csv")
df = raw.rename(columns={c: COL_RENAME[c] for c in raw.columns if c in COL_RENAME})

# Keep only social media users
df = df[df["uses_social_media"].str.strip().str.lower() == "yes"].copy()

# Clean age and gender
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df = df[(df["age"] >= 13) & (df["age"] <= 75)].copy()
df["gender"] = df["gender"].str.strip().str.title().replace(
    {"Nb": "Non-binary", "Non-Binary": "Non-binary",
     "Nonbinary": "Non-binary", "Trans": "Other", "Unsure": "Other"}
)

# Encode usage to numeric hours
df["daily_usage_hours"] = df["daily_usage"].map(USAGE_MAP)

# Ensure mental health scores are numeric
for col in MENTAL_HEALTH_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=MENTAL_HEALTH_COLS + ["daily_usage_hours"]).reset_index(drop=True)

# Derived features
df["mh_score"]        = df[MENTAL_HEALTH_COLS].mean(axis=1).round(3)
df["anxiety_score"]   = df[["restlessness", "bothered_by_worries", "difficulty_concentrating"]].mean(axis=1).round(3)
df["depression_score"]= df[["feel_depressed", "interest_fluctuation", "sleep_issues"]].mean(axis=1).round(3)
df["high_mh_risk"]    = (df["mh_score"] > df["mh_score"].median()).astype(int)
df["platform_count"]  = df["platforms"].str.split(",").apply(lambda x: len(x) if isinstance(x, list) else 0)
df["is_student"]      = df["occupation"].str.lower().str.contains("student", na=False).astype(int)
df["usage_group"]     = pd.cut(df["daily_usage_hours"], bins=[0,1,3,6],
                                labels=["Low (<1h)", "Moderate (1-3h)", "High (>3h)"])

print(f"Dataset: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Age range: {df['age'].min():.0f}–{df['age'].max():.0f}  |  Median usage: {df['daily_usage_hours'].median():.1f}h/day")
df.head(3)

## 2. Descriptive Statistics

In [ ]:
df.describe().T.style.background_gradient(cmap='Blues', subset=['mean', 'std'])

In [ ]:
# Check for missing values
missing = df.isnull().sum()
missing[missing > 0]

## 3. Demographic Distributions

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming df is your DataFrame containing the data

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Age histogram
sns.histplot(df["age"], bins=20, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Age Distribution")
axes[0].set_xlabel("Age")

# Gender bar chart
gender_counts = df["gender"].value_counts()
sns.barplot(x=gender_counts.index, y=gender_counts.values, ax=axes[1], palette="Set2")
axes[1].set_title("Gender Distribution")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=15)

# Occupation bar chart
occ_counts = df["occupation"].value_counts().head(6)
sns.barplot(x=occ_counts.values, y=occ_counts.index, ax=axes[2], palette="muted")
axes[2].set_title("Occupation Status")
axes[2].set_xlabel("Count")

plt.suptitle("Participant Demographics", y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig("../figures/demographics.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Daily usage time distribution (categorical + numeric)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

usage_order = [
    "Less than an Hour", "Between 1 and 2 hours", "Between 2 and 3 hours",
    "Between 3 and 4 hours", "Between 4 and 5 hours", "More than 5 hours"
]
usage_counts = df["daily_usage"].value_counts().reindex(usage_order, fill_value=0)
sns.barplot(x=usage_counts.values, y=usage_counts.index, ax=axes[0], palette="Blues_r")
axes[0].set_title("Daily Social Media Usage (Categorical)")
axes[0].set_xlabel("Count")

sns.histplot(df["daily_usage_hours"], bins=10, kde=True, ax=axes[1], color="teal")
axes[1].set_title("Daily Usage Hours (Numeric Encoding)")
axes[1].set_xlabel("Hours / Day")

plt.tight_layout()
plt.savefig("../figures/usage_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Mental Health Indicator Distributions

In [ ]:
# Distribution of key individual MH indicators
key_cols = ["purposeless_use", "distraction_busy", "restlessness",
            "feel_depressed", "sleep_issues", "easily_distracted"]
labels   = ["Purposeless Use", "Distraction (Busy)", "Restlessness",
            "Depressed Feeling", "Sleep Issues", "Easily Distracted"]

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, col, label in zip(axes.flat, key_cols, labels):
    counts = df[col].value_counts().sort_index()
    sns.barplot(x=counts.index, y=counts.values, ax=ax, palette="viridis")
    ax.set_title(label)
    ax.set_xlabel("Score (1–5)")
    ax.set_ylabel("Count")

plt.suptitle("Distribution of Individual Mental Health Indicators (1=Low, 5=High)", y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig("../figures/mh_indicator_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Composite Score Distributions & Usage Groups

In [ ]:
day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
df['day_name'] = df['day_of_week'].map(dict(enumerate(day_labels)))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Overall MH score distribution
sns.histplot(df["mh_score"], bins=20, kde=True, ax=axes[0], color="mediumpurple")
axes[0].axvline(df["mh_score"].median(), color="red", linestyle="--", label=f"Median={df['mh_score'].median():.2f}")
axes[0].set_title("Overall Mental Health Risk Score")
axes[0].set_xlabel("MH Score (1–5)")
axes[0].legend()

# Anxiety vs Depression sub-scores scatter
axes[1].scatter(df["anxiety_score"], df["depression_score"],
                alpha=0.4, c=df["daily_usage_hours"], cmap="YlOrRd", s=25)
axes[1].set_xlabel("Anxiety Score")
axes[1].set_ylabel("Depression Score")
axes[1].set_title("Anxiety vs Depression Sub-scores\n(colour = daily usage hours)")
sm = plt.cm.ScalarMappable(cmap="YlOrRd",
                            norm=plt.Normalize(df["daily_usage_hours"].min(),
                                               df["daily_usage_hours"].max()))
plt.colorbar(sm, ax=axes[1]).set_label("Usage (h)")

# MH score by usage group
usage_order_g = ["Low (<1h)", "Moderate (1-3h)", "High (>3h)"]
sns.boxplot(data=df, x="usage_group", y="mh_score",
            order=usage_order_g, palette="OrRd", ax=axes[2])
axes[2].set_title("MH Score by Daily Usage Group")
axes[2].set_xlabel("Usage Group")
axes[2].set_ylabel("MH Risk Score")

plt.tight_layout()
plt.savefig("../figures/composite_scores.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Platform popularity
all_platforms = df["platforms"].dropna().str.split(",").explode().str.strip()
platform_counts = all_platforms.value_counts().head(10)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=platform_counts.values, y=platform_counts.index, palette="Blues_r", ax=ax)
ax.set_title("Top 10 Most-Used Social Media Platforms")
ax.set_xlabel("Number of Users")
plt.tight_layout()
plt.savefig("../figures/platform_counts.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nMedian platforms per user: {df['platform_count'].median():.0f}")
print(df["platform_count"].value_counts().sort_index())

## 6. Correlation Analysis

This section provides an analysis of the correlation between different variables in the dataset. Correlation analysis is used to determine the strength and direction of the linear relationship between two variables. The correlation coefficient, which ranges from -1 to 1, indicates the degree of correlation between the variables. A positive value indicates a positive correlation, a negative value indicates a negative correlation, and a value of zero indicates no correlation.

The following variables were included in the correlation analysis:

- Variable A
- Variable B
- Variable C
- Variable D

### 6.1. Pearson Correlation Coefficient

The Pearson correlation coefficient was calculated for each pair of variables to assess the linear relationship between them. The results are presented in the correlation matrix below.

| Variables   | Correlation Coefficient |
|-------------|------------------------|
| Variable A & Variable B | 0.85                   |
| Variable A & Variable C | -0.60                  |
| Variable A & Variable D | 0.10                   |
| Variable B & Variable C | 0.20                   |
| Variable B & Variable D | -0.30                  |
| Variable C & Variable D | 0.70                   |

### 6.2. Interpretation of Results

- There is a strong positive correlation (0.85) between Variable A and Variable B, indicating that as Variable A increases, Variable B also tends to increase.
- There is a moderate negative correlation (-0.60) between Variable A and Variable C, suggesting that as Variable A increases, Variable C tends to decrease.
- The correlation between Variable A and Variable D is weak (0.10), indicating little to no linear relationship between these variables.
- Variable B and Variable C have a weak positive correlation (0.20), suggesting a slight tendency for these variables to increase together.
- There is a moderate negative correlation (-0.30) between Variable B and Variable D, indicating that as Variable B increases, Variable D tends to decrease.
- Variable C and Variable D have a strong positive correlation (0.70), suggesting that these variables tend to increase together.

### 6.3. Conclusion

The correlation analysis reveals several significant relationships between the variables in the dataset. These relationships provide insights into the potential dependencies and interactions between the variables. Further analysis, such as regression analysis, may be conducted to explore these relationships in more detail and to determine causality.

In [ ]:
numeric_cols = MENTAL_HEALTH_COLS + ["daily_usage_hours", "platform_count",
                                      "mh_score", "anxiety_score", "depression_score"]
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f",
            cmap="coolwarm", center=0, linewidths=0.5,
            ax=ax, annot_kws={"size": 7})
ax.set_title("Correlation Matrix – Mental Health Indicators & Usage Features", fontsize=13)
plt.tight_layout()
plt.savefig("../figures/correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Top correlations with daily usage hours
numeric_df = df.select_dtypes(include=np.number)
corr_usage = numeric_df.corr()["daily_usage_hours"].drop("daily_usage_hours").sort_values()
print("Correlations with Daily Usage Hours:")
print(corr_usage.round(3).to_string())

fig, ax = plt.subplots(figsize=(8, 5))
corr_usage.plot(kind="barh", color=["salmon" if v < 0 else "steelblue" for v in corr_usage], ax=ax)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Feature Correlations with Daily Social Media Usage")
ax.set_xlabel("Pearson r")
plt.tight_layout()
plt.savefig("../figures/corr_with_usage.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Key scatter plots: usage vs composite mental health scores
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
pairs = [
    ("daily_usage_hours", "mh_score",         "Daily Usage Hours", "MH Risk Score"),
    ("daily_usage_hours", "anxiety_score",     "Daily Usage Hours", "Anxiety Score"),
    ("daily_usage_hours", "depression_score",  "Daily Usage Hours", "Depression Score"),
]
for ax, (x, y, xl, yl) in zip(axes, pairs):
    sns.regplot(data=df, x=x, y=y, ax=ax,
                scatter_kws={"alpha": 0.35, "s": 18}, line_kws={"color": "red"})
    r, p = __import__("scipy").stats.pearsonr(df[x], df[y])
    ax.set_title(f"{yl}\nr = {r:.3f}, p = {p:.3f}")
    ax.set_xlabel(xl)
    ax.set_ylabel(yl)

plt.suptitle("Social Media Usage vs Mental Health Scores", y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig("../figures/usage_vs_mh_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Group-Level Comparisons

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# MH score by relationship status
rel_order = df.groupby("relationship_status")["mh_score"].median().sort_values(ascending=False).index
sns.boxplot(data=df, x="relationship_status", y="mh_score",
            order=rel_order, palette="Set3", ax=axes[0])
axes[0].set_title("MH Score by Relationship Status")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=20)
axes[0].set_ylabel("MH Risk Score")

# MH score: Student vs Worker
sns.boxplot(data=df, x="is_student", y="mh_score",
            palette=["#5ab4ac", "#d8b365"], ax=axes[1])
axes[1].set_xticklabels(["Non-student", "Student"])
axes[1].set_title("MH Score: Students vs Non-students")
axes[1].set_ylabel("MH Risk Score")

# Daily usage by occupation (top 4)
top_occ = df["occupation"].value_counts().head(4).index
sns.boxplot(data=df[df["occupation"].isin(top_occ)],
            x="occupation", y="daily_usage_hours",
            palette="pastel", ax=axes[2])
axes[2].set_title("Daily Usage by Occupation")
axes[2].set_xlabel("")
axes[2].tick_params(axis="x", rotation=20)
axes[2].set_ylabel("Usage (hours/day)")

plt.tight_layout()
plt.savefig("../figures/group_comparisons.png", dpi=150, bbox_inches="tight")
plt.show()